# Experiment 0.2: Real CRM Opportunity Ranking (Databricks)

Adaptado do kernel Kaggle de Vladimir Belsch para o workspace Databricks do challenge 003.

1. Rode o **LAB 01** (`01_LAB_importando_dados/lab01_carga_csv`) para criar `workspace.crm.*`.
2. Widgets: `catalog_name`, `schema_name`, `data_path` (fallback CSV em `dados/` se a tabela Delta ainda nao existir).
3. Colunas = header dos CSVs. `GTXPro` e normalizado **depois** da carga (o Lab 01 grava o valor cru).
4. Este notebook ranqueia so Won/Lost — o brief pede ferramenta para o pipe **aberto**. Use o score como logica, nao como entrega.
5. Cluster ou serverless → **Run All**.


# Experiment 0.2: Real CRM Opportunity Ranking

This is the second experiment in the Deal Appetite series. Experiment 0.1 showed that a latent ranking can be recovered in a controlled synthetic setting when the data-generating process is known.

This experiment removes that assumption and tests the idea on real CRM opportunity data. The question is narrower: do ordinary opportunity fields contain enough stable signal to rank future won and lost deals on unseen data?

## Objective

Test whether observable CRM opportunity characteristics contain enough signal to produce a useful out-of-sample ranking of sales outcomes.

Unlike Experiment 0.1, the latent appetite variable is not observed or imposed by construction.

The initial empirical objective is therefore narrower:

> Can historical opportunity data produce a ranking that meaningfully separates won and lost opportunities on unseen data?

The observed outcome is:

$$
Y_i =
\begin{cases}
1 & \text{if opportunity } i \text{ is won} \\
0 & \text{if opportunity } i \text{ is lost}
\end{cases}
$$

The model produces a score:

$$
\hat{p}_i = P(Y_i = 1 \mid \mathbf{x}_i)
$$

where $\mathbf{x}_i$ contains only information that would have been available before the final sales outcome was known.

The primary evaluation metric is ROC AUC:

$$
\operatorname{AUC}
=
P
\left(
\hat{p}_{+} > \hat{p}_{-}
\right)
$$

AUC therefore measures whether the model assigns higher scores to won opportunities than to lost opportunities on held-out data.

## 1. Prepare the CRM Dataset

The source data is relational, with opportunity records linked to account,
product, and sales-team tables.

For the initial real-data experiment:

- only resolved opportunities are retained
- `Won` is encoded as $1$
- `Lost` is encoded as $0$
- known join-key inconsistencies are normalized
- post-outcome fields are retained in the dataframe for reference but excluded from modeling
- structural account information is converted into model-friendly features


In [ ]:
from pathlib import Path
import pandas as pd

DATE_COLUMNS = {"engage_date", "close_date"}


def widget_or_default(name, default):
    try:
        dbutils.widgets.text(name, default)
        value = dbutils.widgets.get(name)
        return value if value else default
    except NameError:
        return default


def normalize_frame(frame):
    for col in frame.columns:
        if col in DATE_COLUMNS:
            frame[col] = pd.to_datetime(frame[col], errors="coerce")
    return frame


def resolve_data_path(explicit):
    marker = "sales_pipeline.csv"
    candidates = []
    if explicit:
        candidates.append(Path(explicit))
    try:
        notebook_path = (
            dbutils.notebook.entry_point.getDbutils()
            .notebook()
            .getContext()
            .notebookPath()
            .get()
        )
        parts = [part for part in notebook_path.split("/") if part]
        if "notebook-build-003-lead-scorer" in parts:
            idx = parts.index("notebook-build-003-lead-scorer")
            repo = "/".join(parts[: idx + 1])
            candidates.append(Path("/Workspace") / repo / "dados")
    except Exception:
        pass
    here = Path.cwd()
    candidates.extend([here / "dados", here.parent / "dados", here.parent.parent / "dados"])
    for candidate in candidates:
        if (candidate / marker).exists():
            return str(candidate)
    raise FileNotFoundError("Rode o LAB 01 ou defina data_path para a pasta dados/.")


catalog_name = widget_or_default("catalog_name", "workspace")
schema_name = widget_or_default("schema_name", "crm")
data_path_widget = widget_or_default("data_path", "")

try:
    accounts = spark.table(f"{catalog_name}.{schema_name}.accounts").toPandas()
    products = spark.table(f"{catalog_name}.{schema_name}.products").toPandas()
    sales_pipeline = spark.table(f"{catalog_name}.{schema_name}.sales_pipeline").toPandas()
    sales_teams = spark.table(f"{catalog_name}.{schema_name}.sales_teams").toPandas()
    print(f"loaded Delta {catalog_name}.{schema_name}.*")
except Exception as exc:
    print("Delta indisponivel, lendo CSV:", exc)
    data_path = Path(resolve_data_path(data_path_widget))
    accounts = pd.read_csv(data_path / "accounts.csv")
    products = pd.read_csv(data_path / "products.csv")
    sales_pipeline = pd.read_csv(data_path / "sales_pipeline.csv")
    sales_teams = pd.read_csv(data_path / "sales_teams.csv")
    print(f"loaded CSV from {data_path}")

accounts = normalize_frame(accounts)
products = normalize_frame(products)
sales_pipeline = normalize_frame(sales_pipeline)
sales_teams = normalize_frame(sales_teams)

print({
    "accounts": accounts.shape,
    "products": products.shape,
    "sales_pipeline": sales_pipeline.shape,
    "sales_teams": sales_teams.shape,
})


In [ ]:
# Normalize known join-key inconsistencies (CSV traz GTXPro).

sales_pipeline["product"] = sales_pipeline["product"].replace({
    "GTXPro": "GTX Pro"
})


# Keep only opportunities with known final outcomes.

resolved = sales_pipeline[
    sales_pipeline["deal_stage"].isin(["Won", "Lost"])
].copy()


# Encode binary outcome:
#   Won  = 1
#   Lost = 0

resolved["won"] = (
    resolved["deal_stage"] == "Won"
).astype(int)


# Join account, product, and sales-team information.

model_df = (
    resolved
    .merge(
        accounts,
        on="account",
        how="left"
    )
    .merge(
        products,
        on="product",
        how="left"
    )
    .merge(
        sales_teams,
        on="sales_agent",
        how="left"
    )
)


# Convert parent-company information into a simple structural feature.

model_df["is_subsidiary"] = (
    model_df["subsidiary_of"]
    .notna()
    .astype(int)
)

model_df = model_df.drop(
    columns=["subsidiary_of"]
)


# Convert engagement date to datetime for later temporal features.

model_df["engage_date"] = pd.to_datetime(
    model_df["engage_date"]
)


# Verify required fields after normalization and joins.

assert model_df["series"].notna().all()
assert model_df["sales_price"].notna().all()
assert model_df["won"].notna().all()
assert model_df.isna().sum().sum() == 0


# Inspect the prepared dataset.

print(f"Resolved opportunities: {len(model_df):,}")
print(f"Win rate: {model_df['won'].mean():.3f}")
print(f"Remaining missing values: {model_df.isna().sum().sum():,}")

model_df.head()


## 2. Feature Policy

The baseline model uses only information that could reasonably be available
while an opportunity is active.

The following fields are excluded:

- `opportunity_id`: identifier only
- `deal_stage`: defines the target
- `close_date`: post-outcome information
- `close_value`: realized post-outcome value

The initial experiment retains deal, account, product, and sales-team attributes
that would be known before resolution.


## 3. Feature Construction

The baseline model uses only features that would reasonably be available before an opportunity resolves.

Categorical variables are encoded separately from numerical variables.

The initial feature set includes:

- salesperson identity
- product and product series
- customer account
- account sector and location
- account size characteristics
- salesperson manager and regional office
- engagement timing
- subsidiary status

Post-outcome fields remain excluded.


In [ ]:
# Derive simple temporal features from the engagement date.

model_df["engage_year"] = model_df["engage_date"].dt.year
model_df["engage_month"] = model_df["engage_date"].dt.month
model_df["engage_quarter"] = model_df["engage_date"].dt.quarter


categorical_features = [
    "sales_agent",
    "product",
    "account",
    "sector",
    "office_location",
    "series",
    "manager",
    "regional_office",
]

numeric_features = [
    "year_established",
    "revenue",
    "employees",
    "sales_price",
    "is_subsidiary",
    "engage_year",
    "engage_month",
    "engage_quarter",
]

target = "won"


## 4. Temporal Train/Test Split

A random train/test split can place opportunities from the same time period into both sets.

For this experiment, a temporal split better approximates the real forecasting problem:

> Can historical opportunities be used to rank future opportunities?

The model is therefore trained on earlier deals and evaluated on later deals.

Let $t_i$ denote the engagement date of opportunity $i$.

For a chosen cutoff date $T$:

$$
\text{Train} = \{i : t_i < T\}
$$

and:

$$
\text{Test} = \{i : t_i \ge T\}
$$

No future opportunity is used to train a model that is evaluated on an earlier opportunity.

In [ ]:
# Inspect the temporal coverage of the resolved opportunity data.

print(f"Earliest engagement: {model_df['engage_date'].min()}")
print(f"Latest engagement:   {model_df['engage_date'].max()}")

model_df["engage_date"].describe()

In [ ]:
# Inspect opportunity volume by engagement month.

monthly_counts = (
    model_df
    .set_index("engage_date")
    .resample("ME")
    .size()
)

monthly_counts

### 4.1 Choose the Temporal Cutoff

The resolved opportunities span from October 2016 through December 2017.

For the first temporal validation, we use:

$$
T = \text{2017-09-01}
$$

Opportunities engaged before $T$ are used for training, while opportunities engaged on or after $T$ are reserved for testing.

This preserves chronological ordering and evaluates whether historical opportunities can rank genuinely later opportunities.

In [ ]:
# Define the temporal cutoff.

cutoff_date = pd.Timestamp("2017-09-01")


# Create chronological train and test sets.

train_df = model_df[
    model_df["engage_date"] < cutoff_date
].copy()

test_df = model_df[
    model_df["engage_date"] >= cutoff_date
].copy()


# Inspect split sizes and class balance.

print(f"Train size: {len(train_df):,}")
print(f"Test size:  {len(test_df):,}")

print(f"\nTrain win rate: {train_df['won'].mean():.3f}")
print(f"Test win rate:  {test_df['won'].mean():.3f}")

In [ ]:
print(
    f"Train: {train_df['engage_date'].min().date()} "
    f"to {train_df['engage_date'].max().date()}"
)

print(
    f"Test:  {test_df['engage_date'].min().date()} "
    f"to {test_df['engage_date'].max().date()}"
)

## 5. Baseline Logistic Ranking Model

The first real-data model uses logistic regression as a simple and interpretable baseline.

Categorical features are one-hot encoded, while numerical features are standardized using statistics learned from the training set only.

The model estimates:

$$
\hat{p}_i = P(Y_i = 1 \mid \mathbf{x}_i)
$$

where $Y_i = 1$ indicates a won opportunity.

All preprocessing is fit only on historical training data and then applied to the future test period.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score


# Categorical information available before resolution.

categorical_features = [
    "sales_agent",
    "product",
    "account",
    "sector",
    "office_location",
    "series",
    "manager",
    "regional_office",
]


# Numeric information available before resolution.

numeric_features = [
    "year_established",
    "revenue",
    "employees",
    "sales_price",
    "is_subsidiary",
    "engage_month",
    "engage_quarter",
]


# Separate predictors from the observed outcome.

X_train = train_df[
    categorical_features + numeric_features
]

y_train = train_df["won"]

X_test = test_df[
    categorical_features + numeric_features
]

y_test = test_df["won"]

In [ ]:
# Preprocess categorical and numerical features independently.

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        ),
        (
            "numeric",
            StandardScaler(),
            numeric_features
        )
    ]
)


# Combine preprocessing and logistic regression into
# a single leakage-safe pipeline.

baseline_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000
            )
        )
    ]
)


# Fit using historical opportunities only.

baseline_model.fit(
    X_train,
    y_train
)

In [ ]:
# Generate predicted win probabilities for future opportunities.

predicted_probability = baseline_model.predict_proba(
    X_test
)[:, 1]


# Measure ranking performance on the future holdout period.

auc = roc_auc_score(
    y_test,
    predicted_probability
)

print(f"Temporal test ROC AUC: {auc:.3f}")

### 5.1 Baseline Result

The temporal holdout evaluation produced:

$$
\operatorname{AUC} \approx 0.526
$$

This is only slightly above the random-ranking baseline:

$$
\operatorname{AUC} = 0.5
$$

The baseline model therefore contains very little out-of-sample ranking power on future opportunities.

This does not by itself reject the broader latent-ranking hypothesis.

The result may instead indicate that the available feature set is too coarse, too static, or insufficiently stable across time to recover a useful ordering.

Possible sources of limitation include:

- weak signal in static firmographic and product attributes
- temporal drift between training and test periods
- poor generalization of account or salesperson identity
- missing behavioral information such as interactions, stage transitions, recency, or engagement history
- model misspecification

The next step is to determine where the limited signal originates before introducing more flexible models.

## 6. Feature-Group Ablation

The baseline model combines deal, account, product, salesperson, and temporal information.

To determine where the observed ranking signal originates, we compare several models using progressively broader feature sets while keeping the temporal split and model class fixed.

The experiments are:

1. **Core deal/account/product features**
2. **Core features + salesperson/team information**
3. **Core features + salesperson/team + account identity**

If performance improves substantially after adding a feature group, that group is contributing meaningful out-of-sample ranking information.

In [ ]:
# Train and evaluate a logistic-regression ranking model
# using a specified set of categorical and numeric features.

def evaluate_feature_set(
    name,
    categorical_features,
    numeric_features
):
    features = categorical_features + numeric_features

    X_train_ablation = train_df[features]
    X_test_ablation = test_df[features]

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "categorical",
                OneHotEncoder(handle_unknown="ignore"),
                categorical_features
            ),
            (
                "numeric",
                StandardScaler(),
                numeric_features
            )
        ]
    )

    model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            (
                "classifier",
                LogisticRegression(max_iter=2000)
            )
        ]
    )

    model.fit(
        X_train_ablation,
        y_train
    )

    predicted = model.predict_proba(
        X_test_ablation
    )[:, 1]

    auc = roc_auc_score(
        y_test,
        predicted
    )

    return {
        "model": name,
        "roc_auc": auc
    }

In [ ]:
# Core structural features.

core_categorical = [
    "product",
    "sector",
    "office_location",
    "series",
    "regional_office",
]

core_numeric = [
    "year_established",
    "revenue",
    "employees",
    "sales_price",
    "is_subsidiary",
    "engage_month",
    "engage_quarter",
]


# Add salesperson and management identity.

rep_categorical = core_categorical + [
    "sales_agent",
    "manager",
]


# Add explicit account identity.

account_categorical = rep_categorical + [
    "account",
]

In [ ]:
ablation_results = []

ablation_results.append(
    evaluate_feature_set(
        "Core",
        core_categorical,
        core_numeric
    )
)

ablation_results.append(
    evaluate_feature_set(
        "Core + Rep",
        rep_categorical,
        core_numeric
    )
)

ablation_results.append(
    evaluate_feature_set(
        "Core + Rep + Account",
        account_categorical,
        core_numeric
    )
)


ablation_df = pd.DataFrame(ablation_results)

ablation_df

### 6.1 Ablation Result

The feature-group ablation produced:

$$
\operatorname{AUC}_{\text{Core}} \approx 0.530
$$

$$
\operatorname{AUC}_{\text{Core + Rep}} \approx 0.540
$$

$$
\operatorname{AUC}_{\text{Core + Rep + Account}} \approx 0.525
$$

Salesperson and team information provide a small improvement over the core feature set.

Adding explicit account identity reduces performance on the future holdout period.

This suggests that account identity may encourage memorization of historical account-specific patterns that do not generalize well to later opportunities.

Overall, all three models remain only slightly above random ranking:

$$
\operatorname{AUC} = 0.5
$$

The available static CRM features therefore appear to contain little stable out-of-sample ranking signal.

## 7. Structural-Only Baseline

The previous ablation suggests that salesperson identity provides a small amount of signal, while explicit account identity reduces future-period performance.

To isolate more generalizable deal structure, we remove identity-style features and retain only attributes that describe the opportunity, product, company, region, and engagement timing.

The structural model excludes:

- `sales_agent`
- `manager`
- `account`

and retains only features that can plausibly generalize across unseen opportunities.

In [ ]:
# Structural categorical features only.

structural_categorical = [
    "product",
    "sector",
    "office_location",
    "series",
    "regional_office",
]


# Structural numeric features only.

structural_numeric = [
    "year_established",
    "revenue",
    "employees",
    "sales_price",
    "is_subsidiary",
    "engage_month",
    "engage_quarter",
]

In [ ]:
structural_result = evaluate_feature_set(
    "Structural Only",
    structural_categorical,
    structural_numeric
)

structural_result

### 7.1 Structural Feature Ablation

The structural feature set contains three broad sources of information:

1. product characteristics
2. account firmographics
3. temporal and regional context

We evaluate each group separately to determine whether any one component contains measurable out-of-sample ranking signal.

In [ ]:
# Product information.

product_categorical = [
    "product",
    "series",
]

product_numeric = [
    "sales_price",
]


# Account firmographic information.

firmographic_categorical = [
    "sector",
    "office_location",
]

firmographic_numeric = [
    "year_established",
    "revenue",
    "employees",
    "is_subsidiary",
]


# Temporal and regional context.

context_categorical = [
    "regional_office",
]

context_numeric = [
    "engage_month",
    "engage_quarter",
]

In [ ]:
structural_ablation_results = []

structural_ablation_results.append(
    evaluate_feature_set(
        "Product",
        product_categorical,
        product_numeric
    )
)

structural_ablation_results.append(
    evaluate_feature_set(
        "Firmographics",
        firmographic_categorical,
        firmographic_numeric
    )
)

structural_ablation_results.append(
    evaluate_feature_set(
        "Context",
        context_categorical,
        context_numeric
    )
)

structural_ablation_results.append(
    evaluate_feature_set(
        "All Structural",
        structural_categorical,
        structural_numeric
    )
)


structural_ablation_df = pd.DataFrame(
    structural_ablation_results
)

structural_ablation_df

### 7.2 Structural Ablation Result

The structural feature ablation produced:

$$
\operatorname{AUC}_{\text{Product}} \approx 0.496
$$

$$
\operatorname{AUC}_{\text{Firmographics}} \approx 0.487
$$

$$
\operatorname{AUC}_{\text{Context}} \approx 0.543
$$

$$
\operatorname{AUC}_{\text{All Structural}} \approx 0.530
$$

Product and firmographic features perform at or below random ranking.

The only feature group with measurable out-of-sample signal is temporal and regional context.

This suggests that the dataset contains very little stable opportunity-specific structure that can be used to rank future outcomes.

The weak performance of the combined structural model is therefore more consistent with limited information in the available feature set than with a single poorly chosen feature group.